# LangGraph Module · Day 02 — LangGraph Fundamentals

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Read the core pieces: **`StateGraph`, nodes, edges, checkpointer**
- [ ] Build a simple **2-node agent graph with conditional branching**
- [ ] Understand the **difference vs a plain LangChain chain**

🐍 **Python (30 min):** *TypedDict for LangGraph State* — see `Python/Day02.ipynb`. Today's graph uses exactly that state.

### The one big idea
Day 1's LCEL chain was a **straight pipe**: `prompt | model | parser`, one direction, no memory. A **graph** adds the two things a chain can't do: it carries **shared state** across steps, and it can **branch** (take different paths based on that state). That is the leap from *chain* to *agent*.

## 0. Setup — load the API key

Same as Day 1: this runs against **Google Gemini**, needs `GOOGLE_API_KEY` in a `.env` at the project root.

Install (if needed):
```bash
uv pip install langgraph langchain-google-genai python-dotenv
```

In [1]:
from dotenv import load_dotenv
import os, langgraph

load_dotenv()
print("langgraph imported ✅")
print("GOOGLE_API_KEY:", "loaded ✅" if os.getenv("GOOGLE_API_KEY") else "MISSING ❌ — add it to .env")

langgraph imported ✅
GOOGLE_API_KEY: loaded ✅


## 1. The four pieces of every LangGraph

| Piece | Plain English | Day-1 analogy |
|-------|---------------|---------------|
| **State** | the shared `TypedDict` every node reads & updates | the data flowing through the pipe |
| **Node** | a function `state -> partial update` | one Runnable step |
| **Edge** | a wire saying "after node A, go to node B" | the `|` pipe |
| **Checkpointer** | saves state after each step → memory & resume | *(chains have none)* |

You **declare** the State, **add** nodes, **wire** edges, then **compile** into a runnable graph. Let's build the smallest possible one first.

## 2. Define the State

The state is a `TypedDict` (yesterday's Python topic). `messages` uses the `add_messages` reducer so updates **append** to history instead of overwriting it.

In [2]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    question: str                                  # set once at start (overwrite)
    messages: Annotated[list, add_messages]        # APPEND-reducer (history kept)
    answer: str                                    # filled by a node (overwrite)

print("State schema:", State.__annotations__)

State schema: {'question': <class 'str'>, 'messages': typing.Annotated[list, <function _add_messages_wrapper.<locals>._add_messages at 0x10ef70fe0>], 'answer': <class 'str'>}


`add_messages` is LangGraph's built-in reducer for chat messages — same concept as `operator.add` from the Python notebook, but it also handles message objects and IDs. `START` and `END` are the graph's fixed entry/exit markers.

## 3. A node is just a function: `state -> partial update`

A node reads the state and returns a dict with **only the keys it changed**. LangGraph merges that back (using the reducers).

In [3]:
def greet(state: State) -> dict:
    q = state["question"]
    # return ONLY what changed -- the graph merges it in
    return {
        "messages": [("assistant", f"Got your question: {q}")],
        "answer": f"(stub) you asked: {q}",
    }

# A node is testable in isolation -- it's a plain function
print(greet({"question": "What is a StateGraph?", "messages": [], "answer": ""}))

{'messages': [('assistant', 'Got your question: What is a StateGraph?')], 'answer': '(stub) you asked: What is a StateGraph?'}


## 4. Build the smallest graph: one node

Pattern for every graph:
1. `StateGraph(State)` — make a builder bound to the schema
2. `.add_node(name, fn)` — register node functions
3. `.add_edge(a, b)` — wire them (`START` → node → `END`)
4. `.compile()` — produce a runnable graph (itself a Runnable: `.invoke`/`.stream`)

In [4]:
builder = StateGraph(State)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")     # entry -> greet
builder.add_edge("greet", END)       # greet -> exit

graph = builder.compile()

result = graph.invoke({"question": "What is a StateGraph?", "messages": [], "answer": ""})
print("answer  :", result["answer"])
print("messages:", result["messages"])

answer  : (stub) you asked: What is a StateGraph?
messages: [AIMessage(content='Got your question: What is a StateGraph?', additional_kwargs={}, response_metadata={}, id='bd30faa8-7072-4160-8b06-95c3f0c34110', tool_calls=[], invalid_tool_calls=[])]


☝️ `graph.invoke(initial_state)` runs `START → greet → END`, applying the reducer to `messages`. The return value is the **final merged state**. Notice it's still the same `.invoke()` interface as a Day-1 chain — a compiled graph is a Runnable too.

## 5. The real deliverable — 2 nodes + a conditional branch

Now the part a plain chain **cannot** do: choose the path at runtime based on state.

Plan:
- **`classify`** node — decides if the question is `"math"` or `"general"` and records it.
- A **conditional edge** — routes to one of two nodes based on that decision.
- **`solve_math`** / **`answer_general`** — two different handler nodes.

```
START -> classify -> (branch) --> solve_math      -> END
                              `-> answer_general   -> END
```

In [5]:
from typing import Annotated, Literal, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class QAState(TypedDict):
    question: str
    kind: str                                  # 'math' or 'general' (set by classify)
    messages: Annotated[list, add_messages]
    answer: str

def classify(state: QAState) -> dict:
    q = state["question"].lower()
    kind = "math" if any(ch.isdigit() for ch in q) else "general"
    return {"kind": kind, "messages": [("assistant", f"classified as: {kind}")]}

def solve_math(state: QAState) -> dict:
    return {"answer": f"[math path] I'd compute: {state['question']}"}

def answer_general(state: QAState) -> dict:
    return {"answer": f"[general path] I'd explain: {state['question']}"}

print("nodes defined ✅")

nodes defined ✅


### The routing function

A conditional edge needs a **router**: a function `state -> next_node_name`. It does **not** change state; it just *reads* it and returns which node to go to next.

In [6]:
def route(state: QAState) -> Literal["solve_math", "answer_general"]:
    # read the decision 'classify' wrote, return the next node's name
    return "solve_math" if state["kind"] == "math" else "answer_general"

print(route({"kind": "math"}), "/", route({"kind": "general"}))

solve_math / answer_general


### Wire it with `add_conditional_edges`

`add_conditional_edges(source, router_fn)` says: *after `source`, call `router_fn(state)`; its return value is the name of the next node.*

In [7]:
builder = StateGraph(QAState)
builder.add_node("classify", classify)
builder.add_node("solve_math", solve_math)
builder.add_node("answer_general", answer_general)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route)   # <- the branch
builder.add_edge("solve_math", END)
builder.add_edge("answer_general", END)

qa_graph = builder.compile()
print("compiled ✅")

compiled ✅


### Run both branches — same graph, different paths

In [8]:
for q in ["What is 12 * 3?", "What is LangGraph?"]:
    out = qa_graph.invoke({"question": q, "kind": "", "messages": [], "answer": ""})
    print(f"Q: {q}")
    print(f"   -> kind={out['kind']!r}  answer={out['answer']!r}\n")

Q: What is 12 * 3?
   -> kind='math'  answer="[math path] I'd compute: What is 12 * 3?"

Q: What is LangGraph?
   -> kind='general'  answer="[general path] I'd explain: What is LangGraph?"



☝️ One graph, two outcomes. The path was chosen **at runtime from the state**. A Day-1 LCEL chain (`a | b | c`) always runs every step in the same fixed order — it has no way to make this choice.

## 6. Visualise the graph

LangGraph can render the structure. Handy for confirming the wiring matches your mental model.

In [9]:
print(qa_graph.get_graph().draw_ascii())

                +-----------+                
                | __start__ |                
                +-----------+                
                      *                      
                      *                      
                      *                      
                +----------+                 
                | classify |                 
                +----------+.                
               ..            ..              
             ..                ..            
           ..                    ..          
+----------------+           +------------+  
| answer_general |           | solve_math |  
+----------------+           +------------+  
               **            **              
                 **        **                
                   **    **                  
                 +---------+                 
                 | __end__ |                 
                 +---------+                 


## 7. The checkpointer — giving the graph memory

So far each `invoke` starts blank. A **checkpointer** saves the state after every step, keyed by a `thread_id`. Re-invoke with the same `thread_id` and the graph **resumes** with prior state — this is how agents get conversation memory and human-in-the-loop pause/resume (Day 5).

Use the in-memory `MemorySaver` for local study; production swaps in a SQLite/Postgres saver — same interface.

In [10]:
from langgraph.checkpoint.memory import MemorySaver

# A small graph that appends a message and counts turns, to SEE memory work
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turns: int

def step(state: ChatState) -> dict:
    n = state.get("turns", 0) + 1
    return {"messages": [("assistant", f"reply #{n}")], "turns": n}

b = StateGraph(ChatState)
b.add_node("step", step)
b.add_edge(START, "step")
b.add_edge("step", END)

mem_graph = b.compile(checkpointer=MemorySaver())   # <- memory enabled
print("checkpointer attached ✅")

checkpointer attached ✅


In [11]:
config = {"configurable": {"thread_id": "user-1"}}

# Three separate invokes on the SAME thread -> state persists across them
mem_graph.invoke({"messages": [("user", "hi")], "turns": 0}, config)
mem_graph.invoke({"messages": [("user", "again")]}, config)
final = mem_graph.invoke({"messages": [("user", "third")]}, config)

print("turns counted across 3 invokes:", final["turns"])      # -> 3, memory worked
print("messages remembered:", len(final["messages"]))
# A different thread_id starts fresh:
other = mem_graph.invoke({"messages": [("user", "hi")], "turns": 0},
                         {"configurable": {"thread_id": "user-2"}})
print("new thread turns:", other["turns"])                    # -> 1, isolated

turns counted across 3 invokes: 3
messages remembered: 6
new thread turns: 1


☝️ Same `thread_id` → state accumulates (`turns` reaches 3). Different `thread_id` → fresh state. **This is what a chain fundamentally cannot do** — chains are stateless; each call is independent.

## 8. Chain vs Graph — the honest comparison

| | **LCEL chain** (Day 1) | **LangGraph `StateGraph`** (today) |
|--|--|--|
| Flow | fixed straight line `a \| b \| c` | nodes + edges, can branch & loop |
| Shared state | no — each step's output is next step's input | yes — a `TypedDict` every node reads/updates |
| Branching | no | `add_conditional_edges` routes at runtime |
| Memory across calls | no (stateless) | yes, via **checkpointer** + `thread_id` |
| Human-in-the-loop | no | yes (interrupt/resume — Day 5) |
| Same interface | `.invoke / .stream / .batch` | `.invoke / .stream / .batch` (still a Runnable) |

**Takeaway:** a chain is the right tool for a fixed transform. The moment you need *"it depends"* logic, accumulating state, or memory, you reach for a graph. LangGraph keeps the Runnable interface, so everything you learned Day 1 still applies on top.

## 9. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **`StateGraph(State)`** | builder bound to a `TypedDict` schema | declares the shared state's shape |
| **node** | `state -> partial update` function | one unit of work; merged via reducers |
| **edge** | `add_edge(a, b)` | fixed "go from a to b" wiring |
| **conditional edge** | `add_conditional_edges(src, router)` | pick the next node from state → branching |
| **`compile()`** | turns the builder into a Runnable graph | gives `.invoke/.stream/.batch` |
| **checkpointer** | `MemorySaver()` + `thread_id` | persists state → memory, resume, HITL |

✅ **Next (LangGraph Day 3):** add persistent memory (`SqliteSaver`) and build a multi-turn conversation loop on top of today's graph.